In [9]:
# ==============================================================================
# IMPORTAÇÃO DE BIBLIOTECAS
# ==============================================================================
import os
from dotenv import load_dotenv

import pandas as pd
import psycopg2 as pg
import sqlalchemy
from sqlalchemy import create_engine
import panel as pn

# Inicializa as extensões do Panel para tabelas e notificações
pn.extension()
pn.extension('tabulator')
pn.extension(notifications=True)

# Carrega as variáveis do arquivo .env
load_dotenv()

# Lê as credenciais para o banco de dados
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')

# CONEXÃO COM O BANCO DE DADOS (POSTGRESQL)
# Conexão nativa usando Psycopg2 (usada para cursores, INSERTS, UPDATES e DELETES)
con = pg.connect(
    host=DB_HOST, 
    dbname=DB_NAME, 
    user=DB_USER, 
    password=DB_PASS
)

# String e Engine do SQLAlchemy (usada principalmente pelo Pandas para SELECTS)
cnx = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}'
engine = create_engine(cnx)

NotificationArea(notifications=[Notification(_destroyed=F...])

In [13]:
# ==============================================================================
# MÓDULO DE ACESSO
# ==============================================================================
# Variável para ignorar campos vazios na consulta
flag = ''

import datetime

# Widgets de Identificação
id_usr = pn.widgets.IntInput(name="ID do Usuário (Para Alterar/Excluir)", value=0, start=0)
cpf_usr = pn.widgets.TextInput(name="CPF (Apenas números)", value='', max_length=11, sizing_mode='stretch_width')
nome_usr = pn.widgets.TextInput(name="Nome Completo", value='', sizing_mode='stretch_width')
email_usr = pn.widgets.TextInput(name="E-mail", value='', sizing_mode='stretch_width')
senha_usr = pn.widgets.PasswordInput(name="Senha de Acesso", value='', sizing_mode='stretch_width')
nascimento_usr = pn.widgets.DatePicker(name="Data de Nascimento", value=datetime.date(1990, 1, 1), sizing_mode='stretch_width')

# Widgets de Endereço
cep_usr = pn.widgets.IntInput(name="CEP", value=0, sizing_mode='stretch_width')
num_usr = pn.widgets.IntInput(name="Nº", value=0, sizing_mode='stretch_width')
bairro_usr = pn.widgets.TextInput(name="Bairro", value='', sizing_mode='stretch_width')
cidade_usr = pn.widgets.TextInput(name="Cidade", value='', sizing_mode='stretch_width')
estado_usr = pn.widgets.TextInput(name="Estado (UF)", value='', max_length=2, sizing_mode='stretch_width')

# Botões de Ação
btn_consultar_usr = pn.widgets.Button(name='Consultar Usuários', button_type='primary')
btn_inserir_usr = pn.widgets.Button(name='Inserir Novo', button_type='success')
btn_atualizar_usr = pn.widgets.Button(name='Atualizar Dados', button_type='warning')
btn_excluir_usr = pn.widgets.Button(name='Excluir Usuário', button_type='danger')

def queryAllUsuarios():
    """Consulta os utilizadores cadastrados. Oculta a senha por segurança."""
    query = """
        SELECT id_usuario, cpf, nome_completo, email, data_nascimento, cidade, estado 
        FROM usuario 
        ORDER BY id_usuario;
    """
    df = pd.read_sql_query(query, cnx)
    
    # Formata a data para um padrão de leitura mais amigável
    if not df.empty and 'data_nascimento' in df.columns:
        df['data_nascimento'] = pd.to_datetime(df['data_nascimento']).dt.strftime('%d/%m/%Y')
        
    # Mapeamento dos nomes técnicos do banco para exibição na interface gráfica
    titulos_usuarios = {
        'id_usuario': 'ID',
        'cpf': 'CPF',
        'nome_completo': 'Nome Completo',
        'email': 'E-mail',
        'data_nascimento': 'Data de Nascimento',
        'cidade': 'Cidade',
        'estado': 'UF'
    }
        
    return pn.widgets.Tabulator(
        df, 
        titles=titulos_usuarios,
        layout='fit_data_stretch', 
        pagination='remote', 
        page_size=10
    )

def on_inserir_usuario():
    try:
        cursor = con.cursor()
        query = """
            INSERT INTO usuario (cpf, nome_completo, email, senha, data_nascimento, numero_endereco, cep, bairro, estado, cidade) 
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
        """
        valores = (
            cpf_usr.value, nome_usr.value, email_usr.value, senha_usr.value, 
            nascimento_usr.value, num_usr.value, cep_usr.value, bairro_usr.value, 
            estado_usr.value, cidade_usr.value
        )
        cursor.execute(query, valores)
        con.commit()
        cursor.close()
        pn.state.notifications.success("Usuário inserido com sucesso!")
        return queryAllUsuarios()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao inserir: {str(e)}', alert_type='danger')

def on_atualizar_usuario():
    try:
        cursor = con.cursor()
        query = """
            UPDATE usuario 
            SET cpf = %s, nome_completo = %s, email = %s, data_nascimento = %s, 
                numero_endereco = %s, cep = %s, bairro = %s, estado = %s, cidade = %s
            WHERE id_usuario = %s;
        """
        # Nota: A senha não é atualizada aqui para simplificar a segurança
        valores = (
            cpf_usr.value, nome_usr.value, email_usr.value, nascimento_usr.value, 
            num_usr.value, cep_usr.value, bairro_usr.value, estado_usr.value, 
            cidade_usr.value, id_usr.value
        )
        cursor.execute(query, valores)
        con.commit()
        cursor.close()
        pn.state.notifications.success("Dados atualizados com sucesso!")
        return queryAllUsuarios()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao atualizar: {str(e)}', alert_type='danger')

def on_excluir_usuario():
    try:
        cursor = con.cursor()
        cursor.execute("DELETE FROM usuario WHERE id_usuario = %s;", (id_usr.value,))
        con.commit()
        cursor.close()
        pn.state.notifications.success("Usuário removido!")
        return queryAllUsuarios()
    except Exception as e:
        con.rollback()
        # Tratamento especial de Foreign Key alertando sobre a necessidade de Soft Delete
        if 'violates foreign key constraint' in str(e).lower():
            erro_msg = "ERRO: Não pode excluir este usuário pois ele já possui tarefas, colheitas ou ferramentas associadas! (Implemente o Soft Delete)."
        else:
            erro_msg = f'Erro ao excluir: {str(e)}'
        return pn.pane.Alert(erro_msg, alert_type='danger')

last_clicks = {'ins': 0, 'atu': 0, 'exc': 0}

def usuario_table_creator(cons, ins, atu, exc):
    global last_clicks
    if ins > last_clicks['ins']:
        last_clicks['ins'] = ins
        return on_inserir_usuario()
    if atu > last_clicks['atu']:
        last_clicks['atu'] = atu
        return on_atualizar_usuario()
    if exc > last_clicks['exc']:
        last_clicks['exc'] = exc
        return on_excluir_usuario()
    # Se for só consultar, retorna a tabela atualizada
    return queryAllUsuarios()

interactive_table_usr = pn.bind(
    usuario_table_creator, 
    btn_consultar_usr.param.clicks, 
    btn_inserir_usr.param.clicks, 
    btn_atualizar_usr.param.clicks, 
    btn_excluir_usr.param.clicks
)

# Montagem do Layout: Formulário de 2 colunas à esquerda e a tabela à direita
modulo_acessos = pn.Row(
    pn.Column(
        '### 👤 Gestão de Usuários',
        id_usr,
        pn.Row(cpf_usr, nascimento_usr),
        nome_usr,
        email_usr,
        senha_usr,
        '#### 📍 Endereço',
        pn.Row(cep_usr, num_usr),
        pn.Row(cidade_usr, estado_usr),
        bairro_usr,
        pn.Row(btn_inserir_usr, btn_consultar_usr),
        pn.Row(btn_atualizar_usr, btn_excluir_usr),
        width=400
    ),
    pn.Column(
        "## 📋 Listagem de Participantes (Core)",
        interactive_table_usr,
        margin=(10, 0, 0, 20)
    )
)

modulo_acessos

Row
    [0] Column(width=400)
        [0] Markdown(str)
        [1] IntInput(label='ID do Usuário (..., name='ID do Usuário (..., start=0)
        [2] Row
            [0] TextInput(label='CPF (Apenas números)', max_length=11, name='CPF (Apenas números)', sizing_mode='stretch_width')
            [1] DatePicker(label='Data de Nascimento', name='Data de Nascimento', sizing_mode='stretch_width', value=datetime.date(1990, 1, 1))
        [3] TextInput(label='Nome Completo', name='Nome Completo', sizing_mode='stretch_width')
        [4] TextInput(label='E-mail', name='E-mail', sizing_mode='stretch_width')
        [5] PasswordInput(label='Senha de Acesso', name='Senha de Acesso', sizing_mode='stretch_width')
        [6] Markdown(str)
        [7] Row
            [0] IntInput(label='CEP', name='CEP', sizing_mode='stretch_width')
            [1] IntInput(label='Nº', name='Nº', sizing_mode='stretch_width')
        [8] Row
            [0] TextInput(label='Cidade', name='Cidade', sizing_mode='stretch_width')
            [1] TextInput(label='Estado (UF)', max_length=2, name='Estado (UF)', sizing_mode='stretch_width')
        [9] TextInput(label='Bairro', name='Bairro', sizing_mode='stretch_width')
        [10] Row
            [0] Button(button_type='success', color='success', label='Inserir Novo', name='Inserir Novo')
            [1] Button(button_type='primary', color='primary', label='Consultar Usuários', name='Consultar Usuários')
        [11] Row
            [0] Button(button_type='warning', color='warning', label='Atualizar Dados', name='Atualizar Dados')
            [1] Button(button_type='danger', color='danger', label='Excluir Usuário', name='Excluir Usuário')
    [1] Column(margin=(10, 0, 0, 20))
        [0] Markdown(str)
        [1] ParamFunction(function, _pane=Tabulator, defer_load=False)

In [11]:
# ==============================================================================
# MÓDULO DE CULTIVO
# ==============================================================================

# Variável para ignorar campos vazios na consulta
flag = ''

import datetime
import pandas as pd
import panel as pn

# Widgets para o Canteiro
id_canteiro = pn.widgets.IntInput(name="ID do Canteiro (Consultar/Alterar/Excluir)", value=0, start=0, sizing_mode='stretch_width')
status = pn.widgets.TextInput(name="Status", value='', placeholder='Digite o status', sizing_mode='stretch_width')
comprimento = pn.widgets.FloatInput(name="Comprimento (metros)", value=1.0, start=0.1, sizing_mode='stretch_width')
largura = pn.widgets.FloatInput(name="Largura (metros)", value=1.0, start=0.1, sizing_mode='stretch_width')
tipo_solo = pn.widgets.TextInput(name="Tipo de Solo", value='Orgânico', placeholder='Ex: Argiloso, Orgânico', sizing_mode='stretch_width')
id_horta = pn.widgets.IntInput(name="ID da Horta Vinculada", value=1, start=1, sizing_mode='stretch_width')

# Widgets para a Cultura associada ao canteiro
nome_comum = pn.widgets.TextInput(name="🥬 Nome da Cultura (Para Plantio)", value='', placeholder='Ex: Alface, Tomate', sizing_mode='stretch_width')
quantidade_plantada = pn.widgets.IntInput(name="🔢 Quantidade Plantada", value=10, start=1, sizing_mode='stretch_width')
ciclo_dias = pn.widgets.IntInput(name="⏳ Ciclo de Crescimento (Dias)", value=45, start=1, sizing_mode='stretch_width')
data_plantio = pn.widgets.DatePicker(name="📅 Data do Plantio", value=datetime.date.today(), sizing_mode='stretch_width')

# Botões de Ação
btn_consultar_cant = pn.widgets.Button(name='Consultar Painel', button_type='primary')
btn_inserir_cant = pn.widgets.Button(name='Inserir Canteiro + Planta', button_type='success')
btn_atualizar_cant = pn.widgets.Button(name='Atualizar Dados', button_type='warning')
btn_excluir_cant = pn.widgets.Button(name='Remover Registro', button_type='danger')

def queryAllCanteiros():
    """Descobre o nome da coluna dinamicamente e faz o JOIN imune a erros de acentuação."""
    
    # 1. Pega o nome exato da coluna direto do banco
    cursor = con.cursor()
    cursor.execute("SELECT * FROM canteiro LIMIT 0;")
    col_loc = cursor.description[1][0] # Captura dinamicamente a 2ª coluna
    cursor.close()
    
    # 2. Injeta o nome capturado de forma segura na query
    query = f"""
        SELECT 
            cant.id_canteiro, 
            cant.status AS status, 
            cant.comprimento, 
            cant.largura, 
            cant.tipo_solo,
            cult.nome_comum AS cultura_plantada,
            cult.quantidade_plantada,
            cult.ciclo_dias,
            cult.data_plantio
        FROM canteiro cant
        LEFT JOIN cultura cult ON cant.id_canteiro = cult.id_canteiro
        ORDER BY cant.id_canteiro;
    """
    df = pd.read_sql_query(query, cnx)
    
    # Cálculo em tempo de execução do Atributo Derivado (Requisito READ)
    if not df.empty and 'data_plantio' in df.columns:
        df['data_plantio'] = pd.to_datetime(df['data_plantio'])
        df['data_colheita_prevista'] = df.apply(
            lambda row: row['data_plantio'] + datetime.timedelta(days=int(row['ciclo_dias'])) 
            if pd.notnull(row['data_plantio']) and pd.notnull(row['ciclo_dias']) else None, 
            axis=1
        )
        
        # Formata as colunas de data para exibição limpa e trata canteiros sem culturas plantadas
        df['data_plantio'] = df['data_plantio'].dt.strftime('%d/%m/%Y').fillna('Sem plantio')
        df['data_colheita_prevista'] = df['data_colheita_prevista'].dt.strftime('%d/%m/%Y').fillna('-')
        
    titulos_cultivo = {
        'id_canteiro': 'ID',
        'status': 'Status',
        'comprimento': 'Comprimento (m)',
        'largura': 'Largura (m)',
        'tipo_solo': 'Tipo de Solo',
        'cultura_plantada': 'Cultura',
        'quantidade_plantada': 'Qtd. Plantada',
        'ciclo_dias': 'Ciclo (Dias)',
        'data_plantio': 'Data do Plantio',
        'data_colheita_prevista': 'Previsão de Colheita'
    }
        
    return pn.widgets.Tabulator(
        df, 
        titles=titulos_cultivo,
        layout='fit_data_stretch',
        pagination='remote',
        page_size=10
    )

def on_inserir_canteiro():
    """Insere o canteiro e, se houver cultura informada, realiza o plantio (Requisito CREATE)."""
    try:
        cursor = con.cursor()
        
        # Descobre o nome exato da coluna de localização no banco para evitar UndefinedColumn
        cursor.execute("SELECT * FROM canteiro LIMIT 0;")
        col_loc = cursor.description[1][0]
        
        # 1. Insere o Canteiro físico e recupera o ID gerado pelo SERIAL
        query_cant = f'INSERT INTO canteiro (status, comprimento, largura, tipo_solo, id_horta) VALUES (%s, %s, %s, %s, %s) RETURNING id_canteiro;'
        cursor.execute(query_cant, (status.value_input, comprimento.value, largura.value, tipo_solo.value_input, id_horta.value))
        novo_id_canteiro = cursor.fetchone()[0]
        
        # 2. Se o usuário preencheu o nome da cultura, insere na tabela cultura associada
        if nome_comum.value_input.strip() != '':
            query_cult = "INSERT INTO cultura (id_canteiro, nome_comum, quantidade_plantada, ciclo_dias, data_plantio) VALUES (%s, %s, %s, %s, %s);"
            cursor.execute(query_cult, (novo_id_canteiro, nome_comum.value_input, quantidade_plantada.value, ciclo_dias.value, data_plantio.value))
            
        con.commit()
        cursor.close()
        pn.state.notifications.success("Canteiro e Cultura registrados com sucesso!")
        return queryAllCanteiros()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao inserir: {str(e)}', alert_type='danger')

def on_atualizar_canteiro():
    """Atualiza as dimensões e tipo de solo do canteiro físico (Requisito UPDATE)."""
    try:
        cursor = con.cursor()
        cursor.execute("SELECT * FROM canteiro LIMIT 0;")
        col_loc = cursor.description[1][0]
        
        query = f'UPDATE canteiro SET "{col_loc}" = %s, comprimento = %s, largura = %s, tipo_solo = %s WHERE id_canteiro = %s;'
        cursor.execute(query, (localizacao.value_input, comprimento.value, largura.value, tipo_solo.value_input, id_canteiro.value))
        
        con.commit()
        cursor.close()
        pn.state.notifications.success("Dados do canteiro atualizados!")
        return queryAllCanteiros()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao atualizar: {str(e)}', alert_type='danger')

def on_excluir_canteiro():
    """Remove primeiro as culturas associadas para não quebrar a FK e depois remove o canteiro (Requisito DELETE)."""
    try:
        cursor = con.cursor()
        # 1. Remove registros biológicos da cultura atrelados àquele canteiro
        cursor.execute("DELETE FROM cultura WHERE id_canteiro = %s;", (id_canteiro.value,))
        # 2. Remove o canteiro físico
        cursor.execute("DELETE FROM canteiro WHERE id_canteiro = %s;", (id_canteiro.value,))
        
        con.commit()
        cursor.close()
        pn.state.notifications.success("Canteiro e culturas removidos!")
        return queryAllCanteiros()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao excluir: {str(e)}', alert_type='danger')

def canteiro_table_creator(cons, ins, atu, exc):
    if ins: return on_inserir_canteiro()
    if atu: return on_atualizar_canteiro()
    if exc: return on_excluir_canteiro()
    return queryAllCanteiros()

interactive_table_canteiro = pn.bind(
    canteiro_table_creator,
    btn_consultar_cant, btn_inserir_cant, btn_atualizar_cant, btn_excluir_cant
)

modulo_cultivo_completo = pn.Row(
    pn.Column(
        '### 🥬 Infraestrutura Física',
        id_canteiro, status, pn.Row(comprimento, largura), tipo_solo, id_horta,
        '### 🍓 Produção Biológica (Opcional para novos)',
        nome_comum, pn.Row(quantidade_plantada, ciclo_dias), data_plantio,
        pn.Row(btn_consultar_cant, btn_inserir_cant),
        pn.Row(btn_atualizar_cant, btn_excluir_cant),
        width=450,
        sizing_mode='fixed'
    ),
    pn.Column(
        "## 📊 Painel de Monitoramento Agrícola (Core)",
        interactive_table_canteiro,
        margin=(10, 0, 0, 40),
        sizing_mode='stretch_width'
    ),
    sizing_mode='stretch_width'
)

modulo_cultivo_completo

Row(sizing_mode='stretch_width')
    [0] Column(sizing_mode='fixed', width=450)
        [0] Markdown(str)
        [1] IntInput(label='ID do Canteiro (..., name='ID do Canteiro (..., sizing_mode='stretch_width', start=0)
        [2] TextInput(label='Status', name='Status', placeholder='Digite o status', sizing_mode='stretch_width')
        [3] Row
            [0] FloatInput(label='Comprimento (metros)', name='Comprimento (metros)', sizing_mode='stretch_width', start=0.1, value=1.0)
            [1] FloatInput(label='Largura (metros)', name='Largura (metros)', sizing_mode='stretch_width', start=0.1, value=1.0)
        [4] TextInput(label='Tipo de Solo', name='Tipo de Solo', placeholder='Ex: Argiloso, Orgânico', sizing_mode='stretch_width', value='Orgânico')
        [5] IntInput(label='ID da Horta Vinculada', name='ID da Horta Vinculada', sizing_mode='stretch_width', start=1, value=1)
        [6] Markdown(str)
        [7] TextInput(label='🥬 Nome da Cultura (..., name='🥬 Nome da Cultura (..., placeholder='Ex: Alface, Tomate', sizing_mode='stretch_width')
        [8] Row
            [0] IntInput(label='🔢 Quantidade Plantada', name='🔢 Quantidade Plantada', sizing_mode='stretch_width', start=1, value=10)
            [1] IntInput(label='⏳ Ciclo de Crescimento (..., name='⏳ Ciclo de Crescimento (..., sizing_mode='stretch_width', start=1, value=45)
        [9] DatePicker(label='📅 Data do Plantio', name='📅 Data do Plantio', sizing_mode='stretch_width', value=datetime.date(2026, 7, 1))
        [10] Row
            [0] Button(button_type='primary', color='primary', label='Consultar Painel', name='Consultar Painel')
            [1] Button(button_type='success', color='success', label='Inserir Canteiro +..., name='Inserir Canteiro +...)
        [11] Row
            [0] Button(button_type='warning', color='warning', label='Atualizar Dados', name='Atualizar Dados')
            [1] Button(button_type='danger', color='danger', label='Remover Registro', name='Remover Registro')
    [1] Column(margin=(10, 0, 0, 40), sizing_mode='stretch_width')
        [0] Markdown(str)
        [1] ParamFunction(function, _pane=Tabulator, defer_load=False)

In [14]:
# ==============================================================================
# MÓDULO DE OPERAÇÕES
# ==============================================================================

# Widgets da Tarefa
id_tarefa = pn.widgets.IntInput(name="ID da Tarefa (Alterar/Excluir)", value=0, start=0, sizing_mode='stretch_width')
tipo_tarefa = pn.widgets.TextInput(name="📋 Tipo/Descrição da Atividade", value='', placeholder='Ex: Adubação Orgânica, Rega Geral, Poda', sizing_mode='stretch_width')
desc_tarefa = pn.widgets.TextInput(name="📝 Descrição Detalhada", value='', placeholder='Descreva a tarefa', sizing_mode='stretch_width')
obs_tarefa = pn.widgets.TextInput(name="📌 Observações", value='', placeholder='Opcional', sizing_mode='stretch_width')
data_prevista = pn.widgets.DatePicker(name="📅 Data Prevista", value=datetime.date.today(), sizing_mode='stretch_width')
status_tarefa = pn.widgets.Select(name="⚙️ Status", options=['Pendente', 'Em Andamento', 'Concluído'], value='Pendente', sizing_mode='stretch_width')
id_usuario_responsavel = pn.widgets.IntInput(name="👤 ID Usuário", value=1, start=1, sizing_mode='stretch_width')
id_canteiro_alvo = pn.widgets.IntInput(name="🥬 ID Canteiro", value=1, start=1, sizing_mode='stretch_width')

# Widgets de Insumos
id_insumo_gasto = pn.widgets.IntInput(name="📦 ID Insumo (0 se nenhum)", value=0, start=0, sizing_mode='stretch_width')
qtd_insumo_gasta = pn.widgets.FloatInput(name="🔢 Qtd Consumida", value=0.0, start=0.0, sizing_mode='stretch_width')

# Botões
btn_consultar_op = pn.widgets.Button(name='Consultar', button_type='primary')
btn_inserir_op = pn.widgets.Button(name='Agendar Atividade', button_type='success')
btn_atualizar_op = pn.widgets.Button(name='Atualizar', button_type='warning')
btn_excluir_op = pn.widgets.Button(name='Cancelar', button_type='danger')

def queryAllOperacoes():
    """Consulta tarefas, responsável, canteiro e insumos consumidos."""
    query = """
        SELECT
            t.id_tarefa,
            t.tipo AS atividade,
            t.descricao,
            t.observacoes,
            t.data_prevista,
            t.status,
            u.nome_completo AS responsavel,
            cant.id_canteiro AS canteiro,
            i.nome AS insumo_usado,
            uso.quantidade AS qtd_gasta
        FROM tarefa t
        JOIN usuario u ON t.id_usuario = u.id_usuario
        JOIN canteiro cant ON t.id_canteiro = cant.id_canteiro
        LEFT JOIN uso ON t.id_tarefa = uso.id_tarefa
        LEFT JOIN insumos i ON uso.id_insumos = i.id_insumos
        ORDER BY t.id_tarefa DESC;
    """
    df = pd.read_sql_query(query, cnx)

    if not df.empty:
        df['insumo_usado'] = df['insumo_usado'].fillna('Nenhum')
        df['qtd_gasta'] = df['qtd_gasta'].fillna(0.0)
        df['descricao'] = df['descricao'].fillna('')
        df['observacoes'] = df['observacoes'].fillna('')
        if 'data_prevista' in df.columns:
            df['data_prevista'] = pd.to_datetime(df['data_prevista']).dt.strftime('%d/%m/%Y')

    titulos_operacoes = {
        'id_tarefa': 'ID',
        'atividade': 'Atividade',
        'descricao': 'Descrição',
        'observacoes': 'Observações',
        'data_prevista': 'Data Prevista',
        'status': 'Status',
        'responsavel': 'Responsável',
        'canteiro': 'Canteiro',
        'insumo_usado': 'Insumo Utilizado',
        'qtd_gasta': 'Quantidade Gasta'
    }

    return pn.widgets.Tabulator(
        df,
        titles=titulos_operacoes,
        layout='fit_data_stretch',
        pagination='remote',
        page_size=10
    )

def on_inserir_operacao():
    try:
        cursor = con.cursor()
        query = """
            INSERT INTO tarefa (tipo, descricao, observacoes, data_prevista, status, id_usuario, id_canteiro)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            RETURNING id_tarefa;
        """
        cursor.execute(
            query,
            (
                tipo_tarefa.value,
                desc_tarefa.value,
                obs_tarefa.value,
                data_prevista.value,
                status_tarefa.value,
                id_usuario_responsavel.value,
                id_canteiro_alvo.value,
            )
        )
        nova_id = cursor.fetchone()[0]

        if id_insumo_gasto.value > 0 and qtd_insumo_gasta.value > 0:
            cursor.execute(
                "INSERT INTO uso (id_tarefa, id_insumos, quantidade) VALUES (%s, %s, %s);",
                (nova_id, id_insumo_gasto.value, qtd_insumo_gasta.value)
            )

        con.commit()
        cursor.close()
        pn.state.notifications.success("Tarefa agendada com sucesso!")
        return queryAllOperacoes()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao inserir: {str(e)}', alert_type='danger')


def on_atualizar_operacao():
    try:
        cursor = con.cursor()
        cursor.execute(
            "UPDATE tarefa SET tipo = %s, descricao = %s, observacoes = %s, data_prevista = %s, status = %s, id_usuario = %s, id_canteiro = %s WHERE id_tarefa = %s;",
            (
                tipo_tarefa.value,
                desc_tarefa.value,
                obs_tarefa.value,
                data_prevista.value,
                status_tarefa.value,
                id_usuario_responsavel.value,
                id_canteiro_alvo.value,
                id_tarefa.value,
            )
        )

        cursor.execute("DELETE FROM uso WHERE id_tarefa = %s;", (id_tarefa.value,))
        if id_insumo_gasto.value > 0 and qtd_insumo_gasta.value > 0:
            cursor.execute(
                "INSERT INTO uso (id_tarefa, id_insumos, quantidade) VALUES (%s, %s, %s);",
                (id_tarefa.value, id_insumo_gasto.value, qtd_insumo_gasta.value)
            )

        con.commit()
        cursor.close()
        pn.state.notifications.success("Tarefa atualizada com sucesso!")
        return queryAllOperacoes()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao atualizar: {str(e)}', alert_type='danger')


def on_excluir_operacao():
    try:
        cursor = con.cursor()
        cursor.execute("DELETE FROM uso WHERE id_tarefa = %s;", (id_tarefa.value,))
        cursor.execute("DELETE FROM tarefa WHERE id_tarefa = %s;", (id_tarefa.value,))
        con.commit()
        cursor.close()
        pn.state.notifications.success("Atividade cancelada e removida do cronograma!")
        return queryAllOperacoes()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Erro ao excluir: {str(e)}', alert_type='danger')


last_clicks = {'cons': 0, 'ins': 0, 'atu': 0, 'exc': 0}

def operacao_table_creator(cons, ins, atu, exc):
    global last_clicks
    if ins > last_clicks['ins']:
        last_clicks['ins'] = ins
        return on_inserir_operacao()
    if atu > last_clicks['atu']:
        last_clicks['atu'] = atu
        return on_atualizar_operacao()
    if exc > last_clicks['exc']:
        last_clicks['exc'] = exc
        return on_excluir_operacao()
    if cons > last_clicks['cons']:
        last_clicks['cons'] = cons
    return queryAllOperacoes()

interactive_table_op = pn.bind(
    operacao_table_creator,
    btn_consultar_op.param.clicks,
    btn_inserir_op.param.clicks,
    btn_atualizar_op.param.clicks,
    btn_excluir_op.param.clicks
)

modulo_operacoes = pn.Row(
    pn.Column(
        '### 🛠️ Gestão de Atividades',
        id_tarefa,
        tipo_tarefa,
        desc_tarefa,
        obs_tarefa,
        data_prevista,
        status_tarefa,
        pn.Row(id_usuario_responsavel, id_canteiro_alvo),
        '### 🧺 Consumo de Insumos',
        pn.Row(id_insumo_gasto, qtd_insumo_gasta),
        pn.Row(btn_inserir_op, btn_consultar_op),
        pn.Row(btn_atualizar_op, btn_excluir_op),
        width=400,
        sizing_mode='fixed'
    ),
    pn.Column(
        "## 📅 Cronograma de Manutenção",
        interactive_table_op,
        margin=(10, 0, 0, 20),
        sizing_mode='stretch_width'
    ),
    sizing_mode='stretch_width'
)

modulo_operacoes.servable()

Row(sizing_mode='stretch_width')
    [0] Column(sizing_mode='fixed', width=400)
        [0] Markdown(str)
        [1] IntInput(label='ID da Tarefa (..., name='ID da Tarefa (..., sizing_mode='stretch_width', start=0)
        [2] TextInput(label='📋 Tipo/Descrição d..., name='📋 Tipo/Descrição d..., placeholder='Ex: Adubação Orgânica, ..., sizing_mode='stretch_width')
        [3] TextInput(label='📝 Descrição Detalhada', name='📝 Descrição Detalhada', placeholder='Descreva a tarefa', sizing_mode='stretch_width')
        [4] TextInput(label='📌 Observações', name='📌 Observações', placeholder='Opcional', sizing_mode='stretch_width')
        [5] DatePicker(label='📅 Data Prevista', name='📅 Data Prevista', sizing_mode='stretch_width', value=datetime.date(2026, 7, 1))
        [6] Select(label='⚙️ Status', name='⚙️ Status', options=['Pendente', 'Em Andamento...], sizing_mode='stretch_width', value='Pendente')
        [7] Row
            [0] IntInput(label='👤 ID Usuário', name='👤 ID Usuário', sizing_mode='stretch_width', start=1, value=1)
            [1] IntInput(label='🥬 ID Canteiro', name='🥬 ID Canteiro', sizing_mode='stretch_width', start=1, value=1)
        [8] Markdown(str)
        [9] Row
            [0] IntInput(label='📦 ID Insumo (..., name='📦 ID Insumo (..., sizing_mode='stretch_width', start=0)
            [1] FloatInput(label='🔢 Qtd Consumida', name='🔢 Qtd Consumida', sizing_mode='stretch_width', start=0.0)
        [10] Row
            [0] Button(button_type='success', color='success', label='Agendar Atividade', name='Agendar Atividade')
            [1] Button(button_type='primary', color='primary', label='Consultar', name='Consultar')
        [11] Row
            [0] Button(button_type='warning', color='warning', label='Atualizar', name='Atualizar')
            [1] Button(button_type='danger', color='danger', label='Cancelar', name='Cancelar')
    [1] Column(margin=(10, 0, 0, 20), sizing_mode='stretch_width')
        [0] Markdown(str)
        [1] ParamFunction(function, _pane=Tabulator, defer_load=False)